In [1]:
%cd /kaggle/working
!rm -rf echo
!git clone --depth 1 https://github.com/hassanimtiaz158/echolyx-MVP.git echo
%cd /kaggle/working/echo
!git rev-parse --short HEAD

/kaggle/working
Cloning into 'echo'...
fatal: unable to access 'https://github.com/hassanimtiaz158/echolyx-MVP.git/': Could not resolve host: github.com
[Errno 2] No such file or directory: '/kaggle/working/echo'
/kaggle/working
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [2]:
%cd /kaggle/working/echo
!pip install -q -r requirements.txt

[Errno 2] No such file or directory: '/kaggle/working/echo'
/kaggle/working
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
from pathlib import Path

want = {"data-echolyx3", "data-echolyx3", "data-echolyx23"}
mounts = {}
for p in Path("/kaggle/input").rglob("*"):
    if p.is_dir() and p.name in want:
        mounts[p.name] = p
for name in sorted(want):
    if name in mounts:
        print(f"OK   {name} -> {mounts[name]}")
    else:
        print(f"MISS {name}  (attach it: Data > Add Input)")

MISS data-echolyx23  (attach it: Data > Add Input)
OK   data-echolyx3 -> /kaggle/input/datasets/malikhasanali/data-echolyx3


In [4]:
import os
from collections import Counter

exts = (".wav", ".mp3", ".flac", ".ogg")
for name, base in sorted(mounts.items()):
    files = []
    for root, dirs, fns in os.walk(base):
        files += [os.path.join(root, f) for f in fns if f.lower().endswith(exts)]
    counts = Counter(os.path.splitext(f)[1].lower() for f in files)
    sample = [os.path.relpath(f, base) for f in files[:3]]
    print(f"\n=== {name}: {len(files)} audio files {dict(counts)}")
    for s in sample:
        print("   ", s)


=== data-echolyx3: 4851 audio files {'.wav': 4809, '.mp3': 40, '.flac': 2}
    fan/fan/test/section_01_target_test_anomaly_0046_f-n_C.wav
    fan/fan/test/section_02_target_test_normal_0009_n-lv_L4.wav
    fan/fan/test/section_00_target_test_normal_0049_m-n_Z.wav


In [5]:
import os
from pathlib import Path

RAW = Path("/kaggle/working/echo/data/raw")
CHK = Path("/kaggle/working/echo/checkpoints")
RAW.mkdir(parents=True, exist_ok=True)
CHK.mkdir(parents=True, exist_ok=True)

def _link(dest: Path, target: Path):
    if target is None or not target.exists():
        print(f"  !! {dest.name}: NOT FOUND")
        return False
    if dest.is_symlink() or os.path.exists(dest):
        if dest.is_symlink():
            os.unlink(dest)
        else:
            print(f"  !! {dest} exists as real file/dir — leaving it")
            return False
    os.symlink(str(target), str(dest), target_is_directory=target.is_dir())
    print(f"  {dest.name} -> {target}")
    return True

def _sub(mount_name: str, *parts: str):
    base = mounts.get(mount_name)
    if base is None:
        return None
    for rel in parts:
        cand = base / rel
        if cand.exists():
            return cand
    return None

ok = True
ok &= _link(RAW / "freesound",    _sub("data-echolyx", "Sound"))
ok &= _link(RAW / "mimii" / "fan",      _sub("data-echolyx", "dev_fan"))
ok &= _link(RAW / "mimii" / "fan_dg",   _sub("data-echolyx2", "fan"))
ok &= _link(RAW / "broken_fans",  _sub("data-echolyx1", "broken_fans"))

# checkpoint: search every mount for the backbone .pth
ckpt = None
for base in mounts.values():
    hits = [p for p in base.rglob("Cnn14*") if p.is_file()]
    if hits:
        ckpt = sorted(hits, key=lambda p: len(p.parts))[0]
        break
ok &= _link(CHK / "Cnn14_mAP=0.431.pth", ckpt)

print("\nAll links OK:", ok)

  !! freesound: NOT FOUND
  !! fan: NOT FOUND
  !! fan_dg: NOT FOUND
  !! broken_fans: NOT FOUND
  Cnn14_mAP=0.431.pth -> /kaggle/input/datasets/malikhasanali/data-echolyx3/Cnn14_mAP0.431.pth

All links OK: False


In [6]:
%cd /kaggle/working/echo
!ls -l data/raw data/raw/mimii checkpoints/Cnn14_mAP=0.431.pth 2>&1

/kaggle/working/echo
ls: cannot access 'data/raw/mimii': No such file or directory
lrwxrwxrwx 1 root root   69 Aug  6 06:37 'checkpoints/Cnn14_mAP=0.431.pth' -> /kaggle/input/datasets/malikhasanali/data-echolyx3/Cnn14_mAP0.431.pth

data/raw:
total 0


In [7]:
%cd /kaggle/working/echo
!python -m src.data.collect --config configs/config.yaml

/kaggle/working/echo
/usr/bin/python3: Error while finding module specification for 'src.data.collect' (ModuleNotFoundError: No module named 'src')


In [8]:
%cd /kaggle/working/echo
!python -m src.train --config configs/config.yaml

/kaggle/working/echo
/usr/bin/python3: Error while finding module specification for 'src.train' (ModuleNotFoundError: No module named 'src')


In [9]:
%cd /kaggle/working/echo
!python -m src.evaluate --config configs/config.yaml

/kaggle/working/echo
/usr/bin/python3: Error while finding module specification for 'src.evaluate' (ModuleNotFoundError: No module named 'src')


In [10]:
!cd /kaggle/working/echo && tar czf /kaggle/working/echolyx_results.tar.gz checkpoints/best.pt checkpoints/final.pt artifacts/

tar: checkpoints/best.pt: Cannot stat: No such file or directory
tar: checkpoints/final.pt: Cannot stat: No such file or directory
tar: artifacts: Cannot stat: No such file or directory
tar: Exiting with failure status due to previous errors
